# 06 — Part-of-Speech Tagging

**Learning objective.** Understand token-level grammatical labels and build an inspectable rule-based baseline with NLTK.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## 🧠 Visual engineering mental model

![Causal mindmap](assets/mindmaps/06_pos_tagging.svg)

Read the map **left → right**, then inspect the control knob above it. The learning goal is to predict how a control change propagates before running code.

## 🎛️ Change map — if you change this, what moves downstream?

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Add **context features** | the same suffix can receive different tags | ambiguous words become easier to disambiguate |
| Change the **tagset granularity** | number of labels changes | task difficulty and downstream detail change |
| Use rules only | behavior is transparent | coverage breaks on lexical/context ambiguity |

### Engineering rule
Change **one control at a time**, predict the direction of the effect, then measure whether reality matches the prediction.

## 🔮 Predict before you run

1. Can the word `process` be noun and verb? What context would change the tag?
2. What error do you expect from a suffix-only tagger on `they process requests`?

Do not scroll to the output until you have an expected answer—even a rough one.

### When to use
Useful when downstream rules or analysis benefit from grammatical roles.

### When not / caution
Do not add POS as a feature automatically if the end model already learns context and POS adds no measured value.

### Debugging lens
For a wrong tag, inspect neighboring tokens and ask what evidence the tagger could actually see.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


POS tagging predicts labels such as noun, verb, adjective and adverb **for each token in context**. It supports parsing, information extraction and rule systems. State-of-the-art taggers are learned models; a rule baseline is still useful for understanding features and errors.

In [2]:
import nltk
patterns=[
 (r'^-?[0-9]+(\.[0-9]+)?$','CD'),
 (r'.*ing$','VBG'),
 (r'.*ed$','VBD'),
 (r'.*ly$','RB'),
 (r'.*(ous|ful|able|ive)$','JJ'),
 (r'^(the|a|an)$','DT'),
 (r'^(is|are|was|were|be)$','VB'),
 (r'.*s$','NNS'),
 (r'.*','NN')]
tagger=nltk.RegexpTagger(patterns)
tokens='The reliable system processed 25 requests quickly'.lower().split()
print(tagger.tag(tokens))

[('the', 'DT'), ('reliable', 'JJ'), ('system', 'NN'), ('processed', 'VBD'), ('25', 'CD'), ('requests', 'NNS'), ('quickly', 'RB')]


In [3]:
print('Failure case:', tagger.tag('they process requests'.split()))
print('Reason: lexical context matters; suffix-only rules cannot infer every grammatical role.')

Failure case: [('they', 'NN'), ('process', 'NNS'), ('requests', 'NNS')]
Reason: lexical context matters; suffix-only rules cannot infer every grammatical role.


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Define POS tagging as contextual token classification
- Use baseline errors to motivate learned contextual models